In [1]:
# === 08_cell_list.ipynb ====================================================
# Cell-list (linked-cell) acceleration of the contact count. Each segment's
# 2r-expanded axis-aligned bounding box is registered into a uniform grid;
# only segment pairs sharing a cell are tested with the exact clamped-Lumelsky
# distance (correctness is independent of the cell size h). We verify the count
# equals brute force and time both methods in two regimes:
#   (A) fixed window (the paper's dense regime): a constant-factor speedup;
#   (B) fixed density, growing window: cell-list time ~ O(n), brute ~ O(n^2).
# Grayscale seaborn figure, 600 dpi, PNG + PDF, no caption. No paths printed.
import os, json, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3", "savefig.dpi": 600})
GRAY, DARK = "0.5", "0.15"
_FIG, _TAB, _DAT = os.path.join("..","results","figures"), os.path.join("..","results","tables"), os.path.join("..","data")
for _d in (_FIG,_TAB,_DAT): os.makedirs(_d, exist_ok=True)
def savefig(fig,name):
    fig.savefig(os.path.join(_FIG,name+".png"),bbox_inches="tight"); fig.savefig(os.path.join(_FIG,name+".pdf"),bbox_inches="tight"); plt.close(fig)

ELL=np.array([0.2,0.4]); PMIX=np.array([0.5,0.5]); LMAX=float(ELL.max())
twor=0.02; H=0.20                      # 2r and the cell size (tunable)

def seg_seg(P1,P2,Q1,Q2,want_mid=False):
    d1,d2,r=P2-P1,Q2-Q1,P1-Q1
    a=(d1*d1).sum(1);e=(d2*d2).sum(1);f=(d2*r).sum(1);b=(d1*d2).sum(1);c=(d1*r).sum(1)
    den=a*e-b*b; s=np.where(den>1e-12,np.clip((b*f-c*e)/den,0,1),0.0)
    t=np.clip((b*s+f)/e,0,1); s=np.clip((b*t-c)/a,0,1)
    cpA,cpB=P1+s[:,None]*d1,Q1+t[:,None]*d2; dist=np.linalg.norm(cpA-cpB,axis=1)
    return (dist,0.5*(cpA+cpB)) if want_mid else dist

def sample_enlarged(n,rng):
    L=rng.choice(ELL,size=n,p=PMIX); X=rng.uniform(-LMAX/2,1+LMAX/2,size=(n,3))
    v=rng.normal(size=(n,3)); v/=np.linalg.norm(v,axis=1,keepdims=True)
    return X-0.5*L[:,None]*v, X+0.5*L[:,None]*v

def sample_box(n,W,rng):
    L=rng.choice(ELL,size=n,p=PMIX); X=rng.uniform(0,W,size=(n,3))
    v=rng.normal(size=(n,3)); v/=np.linalg.norm(v,axis=1,keepdims=True)
    return X-0.5*L[:,None]*v, X+0.5*L[:,None]*v

def cell_candidates(P1,P2,twor,h):
    n=len(P1); lo=np.minimum(P1,P2)-twor; hi=np.maximum(P1,P2)+twor; origin=lo.min(0)
    c0=np.floor((lo-origin)/h).astype(np.int64); c1=np.floor((hi-origin)/h).astype(np.int64)
    span=c1-c0+1; sx,sy,sz=(int(span[:,0].max()),int(span[:,1].max()),int(span[:,2].max()))
    dims=np.floor((hi.max(0)-origin)/h).astype(np.int64)+2; BX=np.int64(dims[1]*dims[2]); BY=np.int64(dims[2])
    keys,idxs=[],[]
    for dx in range(sx):
        for dy in range(sy):
            for dz in range(sz):
                m=(dx<span[:,0])&(dy<span[:,1])&(dz<span[:,2]); ii=np.nonzero(m)[0]
                cc=c0[ii]+np.array([dx,dy,dz]); keys.append(cc[:,0]*BX+cc[:,1]*BY+cc[:,2]); idxs.append(ii)
    key=np.concatenate(keys); idx=np.concatenate(idxs); o=np.argsort(key,kind="stable"); key,idx=key[o],idx[o]
    pi,pj=[],[]
    for g in np.split(idx,np.nonzero(np.diff(key))[0]+1):
        if g.size>1:
            a,b=np.triu_indices(g.size,1); pi.append(g[a]); pj.append(g[b])
    if not pi: return np.empty(0,int),np.empty(0,int),0
    pi=np.concatenate(pi); pj=np.concatenate(pj)
    lo2=np.minimum(pi,pj).astype(np.int64); hi2=np.maximum(pi,pj).astype(np.int64)
    u=np.unique(lo2*n+hi2); return (u//n).astype(np.int64),(u%n).astype(np.int64),len(u)

def count_cell(P1,P2,twor,h,gate=True):
    i,j,nc=cell_candidates(P1,P2,twor,h)
    if len(i)==0: return 0,0
    dist,mid=seg_seg(P1[i],P2[i],P1[j],P2[j],want_mid=True); hit=dist<=twor
    if gate: hit=hit & np.all((mid>=0)&(mid<=1),axis=1)
    return int(hit.sum()),nc

def count_brute(P1,P2,twor,gate=True):
    n=len(P1); i,j=np.triu_indices(n,1); dist,mid=seg_seg(P1[i],P2[i],P1[j],P2[j],want_mid=True); hit=dist<=twor
    if gate: hit=hit & np.all((mid>=0)&(mid<=1),axis=1)
    return int(hit.sum())

def best(fn,k=3): return min((lambda t0: (fn(), time.time()-t0)[1])(time.time()) for _ in range(k))

# --- (A) fixed window (enlarged seeding + contact-point gate) ---
rowsA=[]
for n in [400,800,1600,3200]:
    rng=np.random.default_rng(n); P1,P2=sample_enlarged(n,rng)
    cb=count_brute(P1,P2,twor,True); cc,nc=count_cell(P1,P2,twor,H,True)
    tb=best(lambda:count_brute(P1,P2,twor,True)); tc=best(lambda:count_cell(P1,P2,twor,H,True))
    rowsA.append(dict(n=n,brute_ms=tb*1e3,cell_ms=tc*1e3,speedup=tb/tc,
                      cand_frac=nc/(n*(n-1)//2),match=int(cb==cc)))
tabA=pd.DataFrame(rowsA); tabA.to_latex(os.path.join(_TAB,"table_celllist_fixedwindow.tex"),index=False,float_format="%.3f")
tabA.to_csv(os.path.join(_DAT,"celllist_fixedwindow.csv"),index=False)
print("(A) fixed window:"); print(tabA.to_string(index=False))

# --- (B) fixed density rho, growing window W=(n/rho)^(1/3) ---
rho=300.0; rowsB=[]
for n in [300,600,1200,2400,4800]:
    W=(n/rho)**(1/3.0); rng=np.random.default_rng(1000+n); P1,P2=sample_box(n,W,rng)
    cb=count_brute(P1,P2,twor,False); cc,nc=count_cell(P1,P2,twor,H,False)
    tb=best(lambda:count_brute(P1,P2,twor,False),2); tc=best(lambda:count_cell(P1,P2,twor,H,False),2)
    rowsB.append(dict(n=n,W=W,brute_ms=tb*1e3,cell_ms=tc*1e3,speedup=tb/tc,cand_per_n=nc/n,match=int(cb==cc)))
tabB=pd.DataFrame(rowsB); tabB.to_latex(os.path.join(_TAB,"table_celllist_fixeddensity.tex"),index=False,float_format="%.3f")
tabB.to_csv(os.path.join(_DAT,"celllist_fixeddensity.csv"),index=False)
print("\n(B) fixed density, growing window:"); print(tabB.to_string(index=False))

n=tabB["n"].values.astype(float); tb=tabB["brute_ms"].values; tc=tabB["cell_ms"].values
fig,ax=plt.subplots(figsize=(5.0,3.4))
ax.loglog(n,tb,"o-",color=DARK,ms=6,label="brute force")
ax.loglog(n,tc,"s-",color=GRAY,ms=6,label="cell list")
ax.loglog(n,tb[0]*(n/n[0])**2,ls=":",color=DARK,lw=1,label=r"$\propto n^2$")
ax.loglog(n,tc[0]*(n/n[0]),ls=":",color=GRAY,lw=1,label=r"$\propto n$")
ax.set_xticks(n); ax.get_xaxis().set_major_formatter(ScalarFormatter()); ax.set_xticks([],minor=True)
ax.set_xlabel(r"$n$ (fixed density, growing window)"); ax.set_ylabel("time / realization (ms)")
ax.legend(frameon=False,fontsize=8)
savefig(fig,"fig_celllist")

(A) fixed window:
   n    brute_ms    cell_ms  speedup  cand_frac  match
 400   36.069155  30.723810 1.173981   0.045714      1
 800  138.125420  30.331850 4.553808   0.045316      1
1600  593.966961 111.119032 5.345322   0.048957      1
3200 2801.146984 392.925978 7.128943   0.047382      1



(B) fixed density, growing window:
   n        W    brute_ms    cell_ms   speedup  cand_per_n  match
 300 1.000000   15.923500  14.499664  1.098198   17.233333      1
 600 1.259921   75.442076  28.168440  2.678248   19.328333      1
1200 1.587401  298.599958  54.102421  5.519161   21.161667      1
2400 2.000000 1632.328987 100.888729 16.179498   21.483750      1
4800 2.519842 6274.338722 228.678465 27.437383   23.289375      1
